In [90]:
from config import ROOT
import pandas as pd 
import os 
from dotenv import load_dotenv
from datetime import datetime, timedelta
from fredapi import Fred
import numpy as np

In [2]:
load_dotenv()
api = os.getenv('FRED_api')

### Convert and organize file, build cleaning and Combining Func for Macro Data

In [37]:
daily = [
    '/Users/tnguyen287/Documents/finance-db/data/raw/macro/T10YIE_10-Year Inflation Expectations (Daily).csv',
    '/Users/tnguyen287/Documents/finance-db/data/raw/macro/TEDRATE_TED Spread (Daily).csv',
    '/Users/tnguyen287/Documents/finance-db/data/raw/macro/DFF_Federal Funds Effective Rate (Daily).csv',
    '/Users/tnguyen287/Documents/finance-db/data/raw/macro/T10Y2Y_Yield Curve 10Y-2Y Spread (Daily).csv',
    '/Users/tnguyen287/Documents/finance-db/data/raw/macro/DCOILWTICO_WTI Crude Oil Price (Daily).csv',
]

monthly = [
    '/Users/tnguyen287/Documents/finance-db/data/raw/macro/MANEMP_ISM PMI (Monthly).csv',
    '/Users/tnguyen287/Documents/finance-db/data/raw/macro/SAHMREALTIME_Sahm Rule (Monthly).csv',
    '/Users/tnguyen287/Documents/finance-db/data/raw/macro/USREC_Recession Indicator (Monthly).csv',
    '/Users/tnguyen287/Documents/finance-db/data/raw/macro/RETAILSMNSA_Retail Sales (Monthly).csv',
    '/Users/tnguyen287/Documents/finance-db/data/raw/macro/CPILFESL_Core CPI (Monthly).csv',
    '/Users/tnguyen287/Documents/finance-db/data/raw/macro/UNRATE_Unemployment Rate (Monthly).csv',
    '/Users/tnguyen287/Documents/finance-db/data/raw/macro/FEDFUNDS_Federal Funds Rate (Monthly).csv',
    '/Users/tnguyen287/Documents/finance-db/data/raw/macro/CPIAUCSL_Headline CPI (Monthly).csv',
    '/Users/tnguyen287/Documents/finance-db/data/raw/macro/UMCSENT_Consumer Sentiment (Monthly).csv',
    '/Users/tnguyen287/Documents/finance-db/data/raw/macro/INDPRO_Industrial Production (Monthly).csv',
    '/Users/tnguyen287/Documents/finance-db/data/raw/macro/M2SL_M2 Money Supply (Monthly).csv',
    '/Users/tnguyen287/Documents/finance-db/data/raw/macro/PCEPI_PCE Inflation (Monthly).csv'
]

quarterly = [
    # Quarterly native
    '/Users/tnguyen287/Documents/finance-db/data/raw/macro/GDPPOT_Potential GDP (Current Quarterly).csv',
    '/Users/tnguyen287/Documents/finance-db/data/raw/macro/GDP_Nominal GDP (Quarterly).csv',
    '/Users/tnguyen287/Documents/finance-db/data/raw/macro/GDPC1_Real GDP (Quarterly).csv',
]

month_quarter = [
    # Monthly — will resample to quarterly
    '/Users/tnguyen287/Documents/finance-db/data/raw/macro/UNRATE_Unemployment Rate (Monthly).csv',
    '/Users/tnguyen287/Documents/finance-db/data/raw/macro/SAHMREALTIME_Sahm Rule (Monthly).csv',
    '/Users/tnguyen287/Documents/finance-db/data/raw/macro/USREC_Recession Indicator (Monthly).csv',
    '/Users/tnguyen287/Documents/finance-db/data/raw/macro/CPIAUCSL_Headline CPI (Monthly).csv',
    '/Users/tnguyen287/Documents/finance-db/data/raw/macro/CPILFESL_Core CPI (Monthly).csv',
    '/Users/tnguyen287/Documents/finance-db/data/raw/macro/PCEPI_PCE Inflation (Monthly).csv',
    '/Users/tnguyen287/Documents/finance-db/data/raw/macro/M2SL_M2 Money Supply (Monthly).csv',
    '/Users/tnguyen287/Documents/finance-db/data/raw/macro/FEDFUNDS_Federal Funds Rate (Monthly).csv',

]

daily_quarter = [
    # Daily — will resample to quarterly
    '/Users/tnguyen287/Documents/finance-db/data/raw/macro/T10Y2Y_Yield Curve 10Y-2Y Spread (Daily).csv',
    '/Users/tnguyen287/Documents/finance-db/data/raw/macro/BAMLH0A0HYM2_High Yield Credit Spread (Daily).csv',
    '/Users/tnguyen287/Documents/finance-db/data/raw/macro/DCOILWTICO_WTI Crude Oil Price (Daily).csv',
]

In [38]:
col_mapping = {
    'MANEMP': 'pmi',
    'SAHMREALTIME': 'sahm_rule',
    'USREC': 'recession',
    'RETAILSMNSA': 'retail_sales',
    'CPILFESL': 'core_cpi',
    'UNRATE': 'unrate',
    'CPIAUCSL': 'headline_cpi',
    'UMCSENT': 'consumer_sentiment',
    'INDPRO': 'industrial_production',
    'M2SL': 'm2',
    'PCEPI': 'pce',
    'GDPPOT': 'potential_gdp',
    'GDP': 'nominal_gdp',
    'GDPC1': 'real_gdp',
    'T10YIE': 'inflation_exp',
    'BAMLH0A0HYM2': 'hy_spread',
    'TEDRATE': 'ted_spread',
    'FEDFUNDS': 'fedfunds',
    'DFF': 'interest_rate',
    'T10Y2Y': 'yield_curve',
    'DCOILWTICO': 'wti',
}

def macro_cleaning(file_path):
    
    df = pd.read_csv(file_path)
    filename = os.path.basename(file_path)
    series_id = filename.split('_')[0]

    df.columns = df.columns.str.lower()
    df_clean = df.rename(columns={
        df.columns[0]: 'date',
        df.columns[1]: col_mapping[series_id]
    })
    df_clean['date'] = pd.to_datetime(df_clean['date'])

    return df_clean

In [85]:
def macro_combine(file_paths, clean=True, start_year=None):
    df_final = None 

    for file in file_paths:
        if clean: 
            df_clean = macro_cleaning(file)
        if df_final is None:
            
            df_final = df_clean
        else:
            df_final = pd.merge(df_final, df_clean, on='date', how='outer')

    if start_year is not None:
        df_final = df_final[df_final['date'].dt.year >= start_year]

    return df_final

### Convert Daily and Monthly Data into Quarterly

In [40]:
df_me_to_qe = macro_combine(month_quarter)
df_me_to_qe = df_me_to_qe.set_index("date")
df_me_to_qe_clean = df_me_to_qe.resample('QE').first().reset_index()

In [41]:
df_daily_quarter = macro_combine(daily_quarter)
df_daily_quarter = df_daily_quarter.set_index("date")
df_daily_quarter_clean = df_daily_quarter.resample('QE').first().reset_index()

In [42]:
df_quarter = macro_combine(quarterly)

In [43]:
df_quarter['year'] = df_quarter['date'].dt.year
df_quarter['quarter'] = df_quarter['date'].dt.quarter

df_me_to_qe_clean['year'] = df_me_to_qe_clean['date'].dt.year
df_me_to_qe_clean['quarter'] = df_me_to_qe_clean['date'].dt.quarter

df_daily_quarter_clean['year'] = df_daily_quarter_clean['date'].dt.year
df_daily_quarter_clean['quarter'] = df_daily_quarter_clean['date'].dt.quarter

macro_quarterly_0 = pd.merge(df_quarter, df_daily_quarter_clean, on=['year', 'quarter'], how='outer')
macro_quarterly = pd.merge(macro_quarterly_0, df_me_to_qe_clean, on=['year', 'quarter'], how='outer')

macro_quarterly = macro_quarterly.drop(columns=['date_x', 'date_y'])
macro_quarterly = macro_quarterly.rename(columns={'date': 'date'})

series = macro_quarterly.set_index('date').isnull().sum(axis=1)
na_rows = series[series >= 13].index.year
bad_years = set(na_rows[:-2])
macro_quarterly_clean = macro_quarterly[~macro_quarterly['date'].dt.year.isin(bad_years)]

In [44]:
macro_quarterly_clean = macro_quarterly_clean[['date', 'potential_gdp', 'nominal_gdp', 'real_gdp',
       'fedfunds', 'yield_curve', 'hy_spread', 'wti', 'unrate',
       'sahm_rule', 'recession', 'headline_cpi', 'core_cpi', 'pce', 'm2']]

In [45]:
macro_quarterly_clean.reset_index(drop=True, inplace=True)
macro_quarterly_clean

,date,potential_gdp,nominal_gdp,real_gdp,fedfunds,yield_curve,hy_spread,wti,unrate,sahm_rule,recession,headline_cpi,core_cpi,pce,m2
0,1947-03-31,NaN,243.164,2182.681,NaN,NaN,NaN,NaN,NaN,NaN,0.0,21.480,NaN,NaN,NaN
1,1947-06-30,NaN,245.968,2176.892,NaN,NaN,NaN,NaN,NaN,NaN,0.0,22.000,NaN,NaN,NaN
2,1947-09-30,NaN,249.585,2172.432,NaN,NaN,NaN,NaN,NaN,NaN,0.0,22.230,NaN,NaN,NaN
3,1947-12-31,NaN,259.745,2206.452,NaN,NaN,NaN,NaN,NaN,NaN,0.0,22.910,NaN,NaN,NaN
4,1948-03-31,NaN,265.742,2239.682,NaN,NaN,NaN,NaN,3.4,NaN,0.0,23.680,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
313,2025-06-30,23548.517662,30485.729,23770.976,4.33,0.30,3.50,71.61,4.2,0.27,0.0,320.302,326.467,126.150,21775.5
314,2025-09-30,23680.152477,31098.027,24026.834,4.33,0.48,2.91,66.64,4.3,0.10,0.0,322.169,328.682,126.960,22019.9
315,2025-12-31,23811.223897,31422.526,24055.749,4.09,0.57,2.81,62.59,4.5,0.43,0.0,325.063,331.043,127.871,22250.4
316,2026-03-31,23945.137839,NaN,NaN,3.64,0.72,2.83,57.21,4.3,0.30,0.0,326.588,332.793,128.965,22469.1


### Clean and combining Monthly Macro Data

In [73]:
macro_monthly = macro_combine(monthly)
macro_monthly = macro_monthly[macro_monthly['date'].dt.year >= 1990]

In [76]:
macro_monthly.isna().mean().values.mean()

np.float64(0.005658709106984969)

In [81]:
macro_monthly

,date,pmi,sahm_rule,recession,retail_sales,core_cpi,unrate,fedfunds,headline_cpi,consumer_sentiment,industrial_production,m2,pce
1621,1990-01-01,17797.0,0.13,0.0,NaN,132.100,5.4,8.23,127.500,93.0,61.7290,3166.8,58.553
1622,1990-02-01,17893.0,0.13,0.0,NaN,132.700,5.3,8.24,128.000,89.5,62.2896,3179.2,58.811
1623,1990-03-01,17868.0,0.10,0.0,NaN,133.500,5.2,8.28,128.600,91.3,62.5999,3190.1,59.033
1624,1990-04-01,17845.0,0.13,0.0,NaN,134.000,5.4,8.26,128.900,93.9,62.4359,3201.6,59.157
1625,1990-05-01,17796.0,0.13,0.0,NaN,134.400,5.4,8.18,129.100,90.6,62.6258,3200.6,59.290
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2051,2025-11-01,12593.0,0.43,0.0,640201.0,331.043,4.5,3.88,325.063,51.0,101.0810,22296.5,128.152
2052,2025-12-01,12580.0,0.35,0.0,716924.0,331.814,4.4,3.72,326.031,52.9,101.6113,22386.9,128.576
2053,2026-01-01,12582.0,0.30,0.0,582287.0,332.793,4.3,3.64,326.588,56.4,101.5954,22469.1,128.965
2054,2026-02-01,12576.0,0.27,0.0,563690.0,333.512,4.4,3.64,327.460,56.6,102.3440,22667.3,129.449
